<a href="https://colab.research.google.com/github/Yuva-04/CODSOFT/blob/main/MD_T_BEHRT_A_Transformer_Based_Multi_Disease_Causal_Inference_Framework_Using_Longitudinal_Electronic_Health_Records_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from transformers import BertConfig, BertModel

class MD_T_BEHRT(nn.Module):
    def __init__(self, vocab_size, num_diseases):
        super(MD_T_BEHRT, self).__init__()

        # 1. The Shared BEHRT-Style Encoder
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=288,
            num_hidden_layers=6,
            num_attention_heads=12,
            intermediate_size=512,
            max_position_embeddings=512
        )
        self.shared_encoder = BertModel(config)

        # 2. Disease-Specific Causal Inference Heads
        # UPDATE: We add +2 to the hidden_size to accept Age and Dosage
        self.disease_heads = nn.ModuleDict({
            f'disease_{i}': nn.Sequential(
                nn.Linear(config.hidden_size + 2, 128),
                nn.ReLU(),
                nn.Linear(128, 1),
                nn.Sigmoid()
            ) for i in range(num_diseases)
        })

    # UPDATE: Added age and dosage arguments
    def forward(self, input_ids, age, dosage, attention_mask=None):
        outputs = self.shared_encoder(input_ids=input_ids, attention_mask=attention_mask)
        patient_representation = outputs.pooler_output

        # Ensure age and dosage are properly shaped for concatenation
        if age.dim() == 1:
            age = age.unsqueeze(1)
        if dosage.dim() == 1:
            dosage = dosage.unsqueeze(1)

        # Stitch the Transformer output together with the continuous variables
        combined_representation = torch.cat((patient_representation, age, dosage), dim=1)

        multi_disease_predictions = {}
        for name, head in self.disease_heads.items():
            multi_disease_predictions[name] = head(combined_representation)

        return multi_disease_predictions

# --- Quick Test ---
if __name__ == "__main__":
    VOCAB_SIZE = 5000
    model = MD_T_BEHRT(vocab_size=VOCAB_SIZE, num_diseases=3)
    dummy_input_ids = torch.randint(0, VOCAB_SIZE, (2, 10))
    dummy_attention_mask = torch.ones((2, 10))

    # Simulating a 65-year-old and 30-year-old taking a 50mg and 100mg dose
    dummy_age = torch.tensor([0.65, 0.30])
    dummy_dosage = torch.tensor([0.50, 1.00])

    predictions = model(dummy_input_ids, dummy_age, dummy_dosage, dummy_attention_mask)
    print("Multimodal Model initialized successfully!")

Multimodal Model initialized successfully!


In [3]:
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence

# 1. Load the real data you uploaded to Colab
print("Loading Synthea conditions data...")
conditions_df = pd.read_csv('/content/conditions.csv')

# 2. Build the Medical Vocabulary
# We assign a unique ID to every specific medical condition
unique_codes = conditions_df['CODE'].unique()
vocab = {'[PAD]': 0, '[UNK]': 1}
for i, code in enumerate(unique_codes):
    vocab[code] = i + 2  # Start assigning IDs from 2

REAL_VOCAB_SIZE = len(vocab)
print(f"Total unique medical codes in our vocabulary: {REAL_VOCAB_SIZE}")

# 3. Create Patient Sequences
print("Sorting records chronologically and tokenizing...")
conditions_df['START'] = pd.to_datetime(conditions_df['START'])
conditions_df = conditions_df.sort_values(by=['PATIENT', 'START'])

# We will grab the first 5 unique patients to test the pipeline
patient_ids = conditions_df['PATIENT'].unique()[:5]
subset_df = conditions_df[conditions_df['PATIENT'].isin(patient_ids)]

patient_sequences = []
for patient_id, group in subset_df.groupby('PATIENT'):
    codes = group['CODE'].tolist()
    # Convert text codes to their integer IDs
    token_ids = [vocab.get(code, vocab['[UNK]']) for code in codes]
    patient_sequences.append(torch.tensor(token_ids))

# 4. Pad the Sequences
# Transformers require all sequences in a batch to be the exact same length
real_input_ids = pad_sequence(patient_sequences, batch_first=True, padding_value=vocab['[PAD]'])
real_attention_mask = (real_input_ids != vocab['[PAD]']).long()

print(f"Shape of our real input tensor: {real_input_ids.shape}")

# 5. Run the real data through the MD-T-BEHRT model
print("\nPassing real patient histories through the model...")
# Re-initialize the model with the exact size of our new medical vocabulary
real_model = MD_T_BEHRT(vocab_size=REAL_VOCAB_SIZE, num_diseases=3)

# Generate dummy age and dosage for the initial test run
# These values should be consistent with the batch size of real_input_ids
dummy_age = torch.tensor([0.65] * real_input_ids.shape[0])  # Example: all 65 years old (normalized)
dummy_dosage = torch.tensor([0.50] * real_input_ids.shape[0]) # Example: all 50mg dose (normalized)

real_predictions = real_model(real_input_ids, dummy_age, dummy_dosage, real_attention_mask)

print("\nOutput probabilities for our 5 real patients:")
for disease, pred in real_predictions.items():
    print(f"{disease}: \n{pred.detach().numpy()}")

Loading Synthea conditions data...
Total unique medical codes in our vocabulary: 204
Sorting records chronologically and tokenizing...
Shape of our real input tensor: torch.Size([5, 70])

Passing real patient histories through the model...

Output probabilities for our 5 real patients:
disease_0: 
[[0.4937689 ]
 [0.489677  ]
 [0.4943497 ]
 [0.4719371 ]
 [0.49876478]]
disease_1: 
[[0.49535623]
 [0.49702495]
 [0.48321396]
 [0.49254808]
 [0.4807449 ]]
disease_2: 
[[0.443613  ]
 [0.44769987]
 [0.43609866]
 [0.43572074]
 [0.4564886 ]]


In [4]:
import numpy as np

# Let's define the keywords for the specific chronic diseases we want to predict
# Synthea uses standard medical descriptions, so we look for matching strings
target_conditions = {
    'disease_0': 'Heart failure',
    'disease_1': 'Chronic kidney disease',
    'disease_2': 'Stroke'
}

print("Extracting Ground Truth Labels for our 5 patients...\n")

# We will store the binary labels (1.0 for Yes, 0.0 for No) in a dictionary
true_labels = {
    'disease_0': [],
    'disease_1': [],
    'disease_2': []
}

# Go through the exact same 5 patients we just pushed through the model
for patient_id in patient_ids:
    # Get all the text descriptions of conditions this specific patient had
    patient_records = subset_df[subset_df['PATIENT'] == patient_id]['DESCRIPTION'].astype(str).tolist()

    # Check if the target condition exists anywhere in their history
    for head, keyword in target_conditions.items():
        # Using a simple substring check (case-insensitive)
        has_disease = any(keyword.lower() in record.lower() for record in patient_records)
        # Append 1.0 if they have it, 0.0 if they don't
        true_labels[head].append([1.0] if has_disease else [0.0])

# Convert our lists into PyTorch tensors so we can calculate the math later
tensor_labels = {
    head: torch.tensor(labels, dtype=torch.float32)
    for head, labels in true_labels.items()
}

# Let's see what the actual reality is for these 5 patients
for head, keyword in target_conditions.items():
    print(f"Actual {keyword} Labels: \n{tensor_labels[head].numpy()}\n")

Extracting Ground Truth Labels for our 5 patients...

Actual Heart failure Labels: 
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

Actual Chronic kidney disease Labels: 
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

Actual Stroke Labels: 
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]]



In [5]:
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence

print("Hunting for patients with target conditions...")

# 1. Find the IDs of patients who actually have our target diseases
has_hf = conditions_df[conditions_df['DESCRIPTION'].str.contains('Heart failure', case=False, na=False)]['PATIENT'].unique()
has_ckd = conditions_df[conditions_df['DESCRIPTION'].str.contains('Chronic kidney disease', case=False, na=False)]['PATIENT'].unique()
has_stroke = conditions_df[conditions_df['DESCRIPTION'].str.contains('Stroke', case=False, na=False)]['PATIENT'].unique()

print(f"Found {len(has_hf)} with Heart Failure, {len(has_ckd)} with CKD, {len(has_stroke)} with Stroke.")

# 2. Select a mixed batch of 5 patients
# Let's try to grab one of each (if they exist in your 1K sample), plus some randoms
mixed_patient_ids = list(set(has_hf) | set(has_ckd) | set(has_stroke))[:3]

# Fill the rest of the batch (up to 5) with patients not in our sick list
healthy_patient_ids = [p for p in conditions_df['PATIENT'].unique() if p not in mixed_patient_ids]
selected_patient_ids = mixed_patient_ids + healthy_patient_ids[:(5 - len(mixed_patient_ids))]

print(f"\nSelected 5 patient IDs for our balanced batch.")

# 3. Re-run the label extraction for these specific 5 patients
target_conditions = {'disease_0': 'Heart failure', 'disease_1': 'Chronic kidney disease', 'disease_2': 'Stroke'}
true_labels = {'disease_0': [], 'disease_1': [], 'disease_2': []}

subset_df = conditions_df[conditions_df['PATIENT'].isin(selected_patient_ids)]

for patient_id in selected_patient_ids:
    patient_records = subset_df[subset_df['PATIENT'] == patient_id]['DESCRIPTION'].astype(str).tolist()
    for head, keyword in target_conditions.items():
        has_disease = any(keyword.lower() in record.lower() for record in patient_records)
        true_labels[head].append([1.0] if has_disease else [0.0])

tensor_labels = {head: torch.tensor(labels, dtype=torch.float32) for head, labels in true_labels.items()}

for head, keyword in target_conditions.items():
    print(f"\nNew Balanced {keyword} Labels: \n{tensor_labels[head].numpy()}")

Hunting for patients with target conditions...
Found 40 with Heart Failure, 31 with CKD, 49 with Stroke.

Selected 5 patient IDs for our balanced batch.

New Balanced Heart failure Labels: 
[[1.]
 [1.]
 [1.]
 [0.]
 [0.]]

New Balanced Chronic kidney disease Labels: 
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]]

New Balanced Stroke Labels: 
[[0.]
 [1.]
 [0.]
 [0.]
 [0.]]


In [6]:
import torch.optim as optim
import torch.nn as nn

print("Preparing the training data for our 5 selected patients...")

# 1. Tokenize the sequences for our 5 specific balanced patients
balanced_sequences = []
for patient_id in selected_patient_ids:
    codes = subset_df[subset_df['PATIENT'] == patient_id]['CODE'].tolist()
    token_ids = [vocab.get(code, vocab['[UNK]']) for code in codes]
    balanced_sequences.append(torch.tensor(token_ids))

# Pad them so they form a perfect mathematical grid
train_input_ids = pad_sequence(balanced_sequences, batch_first=True, padding_value=vocab['[PAD]'])
train_attention_mask = (train_input_ids != vocab['[PAD]']).long()

# Generate dummy age and dosage for the training loop, consistent with the batch size
# Assuming these are normalized as in the earlier step
dummy_train_age = torch.tensor([0.65] * train_input_ids.shape[0])
dummy_train_dosage = torch.tensor([0.50] * train_input_ids.shape[0])

# 2. Set up the Optimizer and Loss Function
# Adam is the standard optimizer that updates the weights
optimizer = optim.Adam(real_model.parameters(), lr=0.001)

# BCELoss (Binary Cross Entropy) measures the difference between a probability (0.50) and a true label (1.0 or 0.0)
criterion = nn.BCELoss()

print("\n🚀 Starting Training Loop (10 Epochs)...\n")

# 3. The Training Loop
epochs = 10
for epoch in range(epochs):
    # Set the model to training mode
    real_model.train()

    # Zero out the gradients from the previous step
    optimizer.zero_grad()

    # Step A: Forward Pass (Make a guess)
    predictions = real_model(train_input_ids, dummy_train_age, dummy_train_dosage, train_attention_mask)

    # Step B: Calculate the Loss (How wrong was the guess?)
    total_loss = 0
    for head, keyword in target_conditions.items():
        # Get the model's predictions for this disease
        pred = predictions[head]
        # Get the true labels we extracted earlier
        true_y = tensor_labels[head]

        # Calculate the error and add it to our total loss
        loss = criterion(pred, true_y)
        total_loss += loss

    # Step C: Backward Pass (Calculate the gradients / the direction to adjust weights)
    total_loss.backward()

    # Step D: Optimizer Step (Actually update the model's weights)
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} | Total Loss: {total_loss.item():.4f}")

print("\n✅ Training complete! The model has started learning.")

# Let's see what the model predicts NOW after 10 epochs of learning
real_model.eval()
with torch.no_grad():
    final_predictions = real_model(train_input_ids, dummy_train_age, dummy_train_dosage, train_attention_mask)
    print("\nModel's predictions after training (compare these to your true labels!):")
    for head, keyword in target_conditions.items():
        print(f"\nPredicted {keyword} Probabilities: \n{final_predictions[head].numpy()}")

Preparing the training data for our 5 selected patients...

🚀 Starting Training Loop (10 Epochs)...

Epoch 1/10 | Total Loss: 2.0455
Epoch 2/10 | Total Loss: 1.5297
Epoch 3/10 | Total Loss: 1.2483
Epoch 4/10 | Total Loss: 1.0468
Epoch 5/10 | Total Loss: 0.8063
Epoch 6/10 | Total Loss: 0.6314
Epoch 7/10 | Total Loss: 0.5486
Epoch 8/10 | Total Loss: 0.3025
Epoch 9/10 | Total Loss: 0.3151
Epoch 10/10 | Total Loss: 0.1654

✅ Training complete! The model has started learning.

Model's predictions after training (compare these to your true labels!):

Predicted Heart failure Probabilities: 
[[0.9662986 ]
 [0.9567526 ]
 [0.969416  ]
 [0.05420925]
 [0.08153246]]

Predicted Chronic kidney disease Probabilities: 
[[0.00236957]
 [0.09001611]
 [0.00273146]
 [0.00785542]
 [0.00394343]]

Predicted Stroke Probabilities: 
[[0.02465899]
 [0.82147205]
 [0.0312407 ]
 [0.0077077 ]
 [0.00596212]]


In [7]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

print("Building the Full Dataset Pipeline...")

all_patient_ids = conditions_df['PATIENT'].unique()
all_sequences = []
all_labels = {'disease_0': [], 'disease_1': [], 'disease_2': []}

for pid in all_patient_ids:
    codes = conditions_df[conditions_df['PATIENT'] == pid]['CODE'].tolist()
    token_ids = [vocab.get(code, vocab['[UNK]']) for code in codes]
    all_sequences.append(torch.tensor(token_ids))

    records = conditions_df[conditions_df['PATIENT'] == pid]['DESCRIPTION'].astype(str).tolist()
    for head, keyword in target_conditions.items():
        has_disease = any(keyword.lower() in r.lower() for r in records)
        all_labels[head].append([1.0] if has_disease else [0.0])

X_padded = pad_sequence(all_sequences, batch_first=True, padding_value=vocab['[PAD]'])
attention_masks = (X_padded != vocab['[PAD]']).long()
Y_tensors = {head: torch.tensor(labels, dtype=torch.float32) for head, labels in all_labels.items()}

# UPDATE: Generate safe mock data for historical patients so the pipeline compiles
np.random.seed(42)
all_ages = np.random.uniform(0.18, 0.90, size=len(all_patient_ids))    # Simulated normalized ages (18-90)
all_dosages = np.random.uniform(0.10, 1.00, size=len(all_patient_ids)) # Simulated normalized dosages

# UPDATE: Added ages and dosages to the dataset class
class EHRDataset(Dataset):
    def __init__(self, input_ids, masks, labels, ages, dosages):
        self.input_ids = input_ids
        self.masks = masks
        self.labels = labels
        self.ages = ages
        self.dosages = dosages

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.masks[idx],
            'age': torch.tensor(self.ages[idx], dtype=torch.float32),
            'dosage': torch.tensor(self.dosages[idx], dtype=torch.float32),
            'disease_0': self.labels['disease_0'][idx],
            'disease_1': self.labels['disease_1'][idx],
            'disease_2': self.labels['disease_2'][idx]
        }

# UPDATE: Pass the new arrays into the dataset
dataset = EHRDataset(X_padded, attention_masks, Y_tensors, all_ages, all_dosages)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✅ Pipeline Ready! Training on: {len(train_dataset)} | Testing on: {len(test_dataset)}")

Building the Full Dataset Pipeline...
✅ Pipeline Ready! Training on: 917 | Testing on: 230


In [8]:
import torch.optim as optim
import torch.nn as nn

full_model = MD_T_BEHRT(vocab_size=REAL_VOCAB_SIZE, num_diseases=3)
optimizer = optim.Adam(full_model.parameters(), lr=0.001)
criterion = nn.BCELoss()
epochs = 5

print("🚀 Starting Full Dataset Training Loop...\n")

for epoch in range(epochs):
    full_model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        # UPDATE: Extract age and dosage
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        age = batch['age']
        dosage = batch['dosage']

        # UPDATE: Pass age and dosage to model
        predictions = full_model(input_ids, age, dosage, attention_mask)

        batch_loss = 0
        for head in ['disease_0', 'disease_1', 'disease_2']:
            batch_loss += criterion(predictions[head], batch[head])

        batch_loss.backward()
        optimizer.step()
        total_train_loss += batch_loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    full_model.eval()
    total_test_loss = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            age = batch['age']
            dosage = batch['dosage']

            predictions = full_model(input_ids, age, dosage, attention_mask)

            batch_loss = 0
            for head in ['disease_0', 'disease_1', 'disease_2']:
                batch_loss += criterion(predictions[head], batch[head])
            total_test_loss += batch_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

print("\n✅ Full Dataset Training Complete!")

🚀 Starting Full Dataset Training Loop...

Epoch 1/5 | Train Loss: 0.5614 | Test Loss: 0.6394
Epoch 2/5 | Train Loss: 0.4479 | Test Loss: 0.6626
Epoch 3/5 | Train Loss: 0.4429 | Test Loss: 0.6750
Epoch 4/5 | Train Loss: 0.4406 | Test Loss: 0.6384
Epoch 5/5 | Train Loss: 0.4415 | Test Loss: 0.6518

✅ Full Dataset Training Complete!


In [9]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from scipy.special import logit, expit
import statsmodels.api as sm

print("Initializing the TMLE Pipeline for Heart Failure (Disease 0)...\n")

# 1. Gather all predictions and true labels from the TEST set
full_model.eval()

Q0_predictions = []
true_outcomes = []
patient_embeddings = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        age = batch['age']
        dosage = batch['dosage']

        outputs = full_model.shared_encoder(input_ids, attention_mask)
        embeddings = outputs.pooler_output

        # Ensure age and dosage are properly shaped for concatenation
        if age.dim() == 1:
            age = age.unsqueeze(1)
        if dosage.dim() == 1:
            dosage = dosage.unsqueeze(1)

        # Recreate the combined representation as in the model's forward method
        combined_representation = torch.cat((embeddings, age, dosage), dim=1)

        predictions = full_model.disease_heads['disease_0'](combined_representation)

        Q0_predictions.extend(predictions.squeeze().numpy())
        true_outcomes.extend(batch['disease_0'].squeeze().numpy())
        # Store the combined representation, as it is 'W' in TMLE
        patient_embeddings.extend(combined_representation.numpy())

Q0 = np.array(Q0_predictions)
Y = np.array(true_outcomes)
W = np.array(patient_embeddings)

# 2. Simulate a Treatment Variable (A)
np.random.seed(42)
A = np.random.binomial(1, 0.4, size=len(Y))

# 3. Calculate Propensity Scores (g)
propensity_model = LogisticRegression(max_iter=1000)
propensity_model.fit(W, A)
g_W = propensity_model.predict_proba(W)[:, 1]
g_W = np.clip(g_W, 0.01, 0.99)

# 4. Calculate the "Clever Covariate" (H)
H_AW = (A / g_W) - ((1 - A) / (1 - g_W))

# 5. The Targeting Step (Fluctuation) using statsmodels
# We use GLM with Binomial family to allow continuous [0,1] targets and an offset
Q0_logit = logit(np.clip(Q0, 0.001, 0.999))

# statsmodels GLM for the fluctuation parameter epsilon
# We fit: logit(P(Y=1|A,W)) = Q0_logit + epsilon * H_AW
fluc_mod = sm.GLM(Y, H_AW, family=sm.families.Binomial(), offset=Q0_logit)
res = fluc_mod.fit()
epsilon = res.params[0]

print(f"Calculated Fluctuation Parameter (Epsilon): {epsilon:.4f}")

# 6. Update predictions to get the Targeted Estimates (Q1)
Q1_targeted = expit(Q0_logit + (epsilon * H_AW))

# 7. Calculate the Average Treatment Effect (ATE)
H_1 = 1 / g_W
Q1_treat = expit(Q0_logit + (epsilon * H_1))

H_0 = -1 / (1 - g_W)
Q1_control = expit(Q0_logit + (epsilon * H_0))

ATE = np.mean(Q1_treat - Q1_control)

print(f"\n✅ TMLE Targeting Complete!")
print(f"Estimated Average Treatment Effect (ATE): {ATE:.4f}")

if ATE < 0:
    print("Conclusion: The treatment DECREASES the probability of Heart Failure (Protective).")
else:
    print("Conclusion: The treatment INCREASES the probability of Heart Failure (Harmful).")

Initializing the TMLE Pipeline for Heart Failure (Disease 0)...

Calculated Fluctuation Parameter (Epsilon): -0.0610

✅ TMLE Targeting Complete!
Estimated Average Treatment Effect (ATE): -0.0111
Conclusion: The treatment DECREASES the probability of Heart Failure (Protective).


In [11]:
import pandas as pd
import numpy as np
import torch
import statsmodels.api as sm
from scipy.special import logit, expit
from sklearn.linear_model import LogisticRegression

print("Loading patient medication records...")
medications_df = pd.read_csv('/content/medications.csv')

# UPDATE: Added user_age and user_dosage arguments
def evaluate_treatment_effect(drug_name, disease_name, model, test_dataloader, patient_ids_in_test, user_age=None, user_dosage=None):
    print(f"\n{'-'*50}")
    print(f"🔬 ANALYZING: Effect of '{drug_name}' on '{disease_name}'")
    if user_age and user_dosage:
        print(f"👤 PROFILE: Age: {user_age} | Dose: {user_dosage}mg")
    print(f"{'-'*50}")

    # FIXED: Case-insensitive dictionary
    disease_map = {
        'heart failure': 'disease_0',
        'chronic kidney disease': 'disease_1',
        'stroke': 'disease_2'
    }
    target_head = disease_map.get(disease_name.lower())

    if not target_head:
        print(f"Error: Disease '{disease_name}' not found in model tracking.")
        return

    model.eval()
    Q0_preds, true_y, W_embeddings = [], [], []

    with torch.no_grad():
        for batch in test_dataloader:
            outputs = model.shared_encoder(batch['input_ids'], batch['attention_mask'])
            patient_repr = outputs.pooler_output

            # UPDATE: Override the historical age/dosage if the user inputted specific values
            if user_age and user_dosage:
                # Normalize values for the model
                norm_age = float(user_age) / 100.0
                norm_dose = float(user_dosage) / 1000.0 # Assuming max dose 1000mg
                age = torch.full_like(batch['age'], norm_age)
                dosage = torch.full_like(batch['dosage'], norm_dose)
            else:
                age = batch['age']
                dosage = batch['dosage']

            if age.dim() == 1: age = age.unsqueeze(1)
            if dosage.dim() == 1: dosage = dosage.unsqueeze(1)

            # Stitch the modified data together
            combined_repr = torch.cat((patient_repr, age, dosage), dim=1)
            W_embeddings.extend(combined_repr.numpy())

            Q0_preds.extend(model.disease_heads[target_head](combined_repr).squeeze().numpy())
            true_y.extend(batch[target_head].squeeze().numpy())

    Q0 = np.array(Q0_preds)
    Y = np.array(true_y)
    W = np.array(W_embeddings)

    A = []
    for pid in patient_ids_in_test:
        patient_meds = medications_df[medications_df['PATIENT'] == pid]['DESCRIPTION'].astype(str).tolist()
        took_drug = any(drug_name.lower() in med.lower() for med in patient_meds)
        A.append(1.0 if took_drug else 0.0)
    A = np.array(A)

    if sum(A) == 0:
        print(f"Not enough data: No patients in the test set took '{drug_name}'.")
        return

    propensity_model = LogisticRegression(max_iter=1000, class_weight='balanced')
    propensity_model.fit(W, A)
    g_W = np.clip(propensity_model.predict_proba(W)[:, 1], 0.05, 0.95)

    H_AW = (A / g_W) - ((1 - A) / (1 - g_W))
    Q0_logit = logit(np.clip(Q0, 0.001, 0.999))
    glm_model = sm.GLM(Y, H_AW, family=sm.families.Binomial(), offset=Q0_logit)
    epsilon = glm_model.fit().params[0]

    Q1_treat = expit(Q0_logit + (epsilon * (1 / g_W)))
    Q1_control = expit(Q0_logit + (epsilon * (-1 / (1 - g_W))))
    ATE = np.mean(Q1_treat - Q1_control)

    print(f"➤ Calculated Fluctuation (Epsilon): {epsilon:.4f}")
    print(f"➤ Average Treatment Effect (ATE):   {ATE:.4f}\n")

    if ATE < -0.01:
        print(f"✅ CLINICAL INSIGHT: Protective.")
        print(f"'{drug_name}' is associated with a {abs(ATE)*100:.2f}% DECREASE in the risk of {disease_name}.")
    elif ATE > 0.01:
        print(f"⚠️ CLINICAL INSIGHT: Harmful / Associated Risk.")
        print(f"'{drug_name}' is associated with a {abs(ATE)*100:.2f}% INCREASE in the risk of {disease_name}.")
    else:
        print(f"➖ CLINICAL INSIGHT: Neutral.")
        print(f"'{drug_name}' shows no significant causal effect on the risk of {disease_name}.")

    print(f"{'-'*50}")
    return ATE

Loading patient medication records...


In [12]:
def run_interactive_causal_engine(model, test_dataloader, patient_ids_in_test):
    print("=" * 60)
    print("⚕️  MD-T-BEHRT: INTERACTIVE CAUSAL INFERENCE ENGINE")
    print("=" * 60)
    print("Enter multiple values separated by commas (e.g., Amlodipine, Metformin)")

    # 1. Get manual input from the user
    user_drugs = input("➤ Enter Target Drug(s): ")
    user_diseases = input("➤ Enter Target Disease(s): ")

    # Clean up the inputs into Python lists
    drugs_list = [d.strip() for d in user_drugs.split(',')]
    diseases_list = [d.strip() for d in user_diseases.split(',')]

    print("\n" + "=" * 60)
    print(f"Executing Multi-Disease Causal Analysis...")
    print("=" * 60)

    # 2. Iterate through every combination of drug and disease
    for drug in drugs_list:
        for disease in diseases_list:
            # We call the exact function we defined in the previous step!
            try:
                evaluate_treatment_effect(drug, disease, model, test_dataloader, patient_ids_in_test)
            except Exception as e:
                print(f"\n{'-'*50}")
                print(f"🔬 ANALYZING: Effect of '{drug}' on '{disease}'")
                print(f"{'-'*50}")
                print(f"⚠️ Could not complete analysis: {str(e)}")
                print(f"{'-'*50}")

# Get patient_ids for the test set from the test_dataset created in a previous cell
patient_indices_in_test = test_dataset.indices
patient_ids_in_test = [all_patient_ids[i] for i in patient_indices_in_test]

# Run the interactive engine
run_interactive_causal_engine(full_model, test_loader, patient_ids_in_test)

⚕️  MD-T-BEHRT: INTERACTIVE CAUSAL INFERENCE ENGINE
Enter multiple values separated by commas (e.g., Amlodipine, Metformin)
➤ Enter Target Drug(s): naproxen sodium
➤ Enter Target Disease(s): stroke

Executing Multi-Disease Causal Analysis...

--------------------------------------------------
🔬 ANALYZING: Effect of 'naproxen sodium' on 'stroke'
--------------------------------------------------
➤ Calculated Fluctuation (Epsilon): 0.2735
➤ Average Treatment Effect (ATE):   0.0599

⚠️ CLINICAL INSIGHT: Harmful / Associated Risk.
'naproxen sodium' is associated with a 5.99% INCREASE in the risk of stroke.
--------------------------------------------------


In [13]:
!pip install gradio

In [14]:
import gradio as gr
import contextlib
import io

# UPDATE: Wrapper now accepts age and dosage
def ui_wrapper(drugs_input, diseases_input, age_input, dosage_input):
    drugs_list = [d.strip() for d in drugs_input.split(',') if d.strip()]
    diseases_list = [d.strip() for d in diseases_input.split(',') if d.strip()]

    if not drugs_list or not diseases_list:
        return "⚠️ Please enter at least one drug and one disease."

    final_output = ""

    for drug in drugs_list:
        for disease in diseases_list:
            f = io.StringIO()
            with contextlib.redirect_stdout(f):
                try:
                    # Pass the UI values right into the inference engine
                    evaluate_treatment_effect(drug, disease, full_model, test_loader, patient_ids_in_test, user_age=age_input, user_dosage=dosage_input)
                except Exception as e:
                    print(f"⚠️ Error analyzing {drug} on {disease}: {str(e)}")

            final_output += f.getvalue() + "\n\n"

    return final_output

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Product+Sans:wght@400;700&display=swap');
body, * { font-family: 'Product Sans', sans-serif !important; }
.gradio-container { background-color: #f8f9fa; border-radius: 12px; }
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as causal_dashboard:
    gr.Markdown(
        """
        # ⚕️ MD-T-BEHRT Causal Inference Engine
        ### Multimodal Treatment Evaluator
        Enter patient demographics and medications to generate doubly robust estimates.
        """
    )

    with gr.Row():
        with gr.Column():
            drugs = gr.Textbox(label="Target Drug(s)", placeholder="e.g., Amlodipine", info="Separate multiple with commas")
            diseases = gr.Textbox(label="Target Disease(s)", placeholder="e.g., Heart failure", info="Separate multiple with commas")

            with gr.Row():
                age_val = gr.Number(label="Patient Age", value=65)
                dose_val = gr.Number(label="Dosage (mg)", value=500)

            analyze_btn = gr.Button("Run Causal Analysis", variant="primary")

        with gr.Column():
            output_display = gr.Textbox(label="Clinical Insights & ATE Results", lines=15, interactive=False)

    analyze_btn.click(
        fn=ui_wrapper,
        inputs=[drugs, diseases, age_val, dose_val],
        outputs=output_display
    )

causal_dashboard.launch(share=True, debug=True)

/tmp/ipykernel_4336/2910781427.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as causal_dashboard:
/tmp/ipykernel_4336/2910781427.py:35: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as causal_dashboard:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8936ea1556844f253b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8936ea1556844f253b.gradio.live
